# C6 example 2/4: `TensorProduct`

The physical feature types are the same as notebook 1: three 3D vectors plus two scalars become one 3D vector plus three scalars. We explicitly restrict each 3D vector as $(x,y)\in E_1$ and $z\in A$.

Unlike `WELinear`, a tensor product needs two representation-valued inputs. Here both slots receive the complete same-node feature vector $x$: all three planar vector parts, all three invariant $z$ components, and both scalar features. The first layer therefore learns genuine feature--feature interactions $x\otimes x$; after the pointwise activation, the second layer similarly uses $h\otimes h$. No separate context or geometry vector is introduced.

In [ ]:
import torch
from we3nn import CyclicGroup, nn

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)
G = CyclicGroup(6)
A = G.trivial_representation
E1 = G.standard_representation
regular = G.regular_representation()
input_rep = 3 * E1 + 5 * A
hidden_rep = 2 * regular
output_rep = E1 + 4 * A
print('dimensions:', input_rep.size, 'x', input_rep.size, '->', hidden_rep.size, 'x', hidden_rep.size, '->', output_rep.size)

In [ ]:
def pack_input(vectors, scalars):
    xy = vectors[..., :, :2].reshape(*vectors.shape[:-2], 6)
    return torch.cat((xy, vectors[..., :, 2], scalars), dim=-1)

def unpack_input(x):
    xy = x[..., :6].reshape(*x.shape[:-1], 3, 2)
    vectors = torch.cat((xy, x[..., 6:9].unsqueeze(-1)), dim=-1)
    return vectors, x[..., 9:11]

def unpack_output(y):
    return torch.cat((y[..., :2], y[..., 2:3]), dim=-1), y[..., 3:6]

vectors = torch.tensor([[[1.0, 0.2, -0.4], [-0.3, 0.8, 1.2], [0.5, -0.7, 0.1]]])
scalars = torch.tensor([[0.6, -1.1]])
x = nn.RepresentationTensor(pack_input(vectors, scalars), input_rep)
print('packed input:', x.tensor)

## Step 1: visualize the physical input

The arrows are the three vector-valued node features. Their three $z$ components and the two additional scalar node features are invariant under planar C6. The complete packed feature tensor is supplied to both tensor-product input slots.

In [ ]:
import matplotlib.pyplot as plt

fig_input = plt.figure(figsize=(6, 5), constrained_layout=True)
ax = fig_input.add_subplot(111, projection='3d')
for index, vector in enumerate(vectors[0]):
    ax.quiver(0, 0, 0, *vector.tolist(), color=f'C{index}', linewidth=2, label=f'node vector v{index+1}')
limit = 1.15 * vectors.abs().max().item()
ax.set(xlim=(-limit, limit), ylim=(-limit, limit), zlim=(-limit, limit), xlabel='x', ylabel='y', zlabel='z', title='Initial node vectors')
ax.set_box_aspect((1, 1, 1)); ax.legend()
print('scalar node features:', scalars[0].tolist())
plt.show()

## Step 2: build the tensor-product architecture

The first layer evaluates all allowed finite-group couplings

$$h_o=\sum_p w_p(C_p)_{oij}x_i x_j.$$

This includes scalar--scalar, scalar--vector, and vector--vector interactions. In particular, $E_1\otimes E_1$ produces invariant dot/area-like channels and a frequency-2 irrep. The output layer applies the same construction to the activated regular features, $y=\operatorname{TP}(h,h)$. Because the same tensor occupies both slots, these are self-interactions; using two different nodes would instead give pairwise interactions. The first layer is intentionally purely quadratic; a constant scalar channel or a `WELinear` residual would add constant or linear terms. `internal_weights=True` stores the reduced coefficients inside each layer.

In [ ]:
class TensorProductNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.input_layer = nn.TensorProduct(
            input_rep, input_rep, hidden_rep,
            internal_weights=True, shared_weights=True,
        )
        self.activation = nn.PointActiv(hidden_rep, torch.relu)
        self.output_layer = nn.TensorProduct(
            hidden_rep, hidden_rep, output_rep,
            internal_weights=True, shared_weights=True,
        )

    def forward(self, features):
        # Both arguments carry every vector and scalar input feature.
        h_pre = self.input_layer(features, features)
        h = self.activation(h_pre)
        y = self.output_layer(h, h)
        return y, h_pre, h

model = TensorProductNetwork().eval()
y, h_pre, h = model(x)
print('input TP paths / weights:', len(model.input_layer.instructions), model.input_layer.weight_numel)
print('output TP paths / weights:', len(model.output_layer.instructions), model.output_layer.weight_numel)
print('hidden before PointActiv:', h_pre.tensor)
print('hidden after  PointActiv:', h.tensor)
print('physical output (vector, scalars):', unpack_output(y.tensor))

## Step 3: run all six rotations and test equivariance

Both representation-valued arguments are the same wrapped tensor in this self-interaction example. The wrapper still validates each slot independently, so `TensorProduct` rejects an accidental representation mismatch rather than merely checking the final dimension.

In [ ]:
errors = []
for k, element in enumerate(G.elements):
    x_k = x.transform_fibers(element)
    y_k, _, _ = model(x_k)
    expected_k = y.transform_fibers(element)
    error = (y_k.tensor - expected_k.tensor).abs().max().item()
    errors.append(error)
    torch.testing.assert_close(y_k.tensor, expected_k.tensor, atol=2e-5, rtol=2e-5)
    in_vectors_k, in_scalars_k = unpack_input(x_k.tensor)
    out_vector_k, out_scalars_k = unpack_output(y_k.tensor)
    print(f'rotation {k}: angle={60*k:3d} degrees')
    print('  input vectors :', in_vectors_k[0].tolist())
    print('  input scalars :', in_scalars_k[0].tolist())
    print('  output vector :', out_vector_k[0].tolist())
    print('  output scalars:', out_scalars_k[0].tolist())
    print(f'  max equivariance error: {error:.3e}')

print('maximum over all rotations:', max(errors))

## Step 4: visualize the hidden regular representations

The heatmap makes the internal permutation action visible: each C6 rotation cyclically shifts the components in both regular-representation copies. No harmonic basis is used here because both tensor-product inputs are node features rather than an externally evaluated geometric filter.

In [ ]:
import matplotlib.pyplot as plt

hidden_by_rotation, output_vectors, output_scalars = [], [], []
for element in G.elements:
    y_k, _, h_k = model(x.transform_fibers(element))
    vector_k, scalars_k = unpack_output(y_k.tensor)
    hidden_by_rotation.append(h_k.tensor[0].detach())
    output_vectors.append(vector_k[0].detach())
    output_scalars.append(scalars_k[0].detach())
hidden_by_rotation = torch.stack(hidden_by_rotation).cpu()
output_vectors = torch.stack(output_vectors).cpu()
output_scalars = torch.stack(output_scalars).cpu()
angles_deg = torch.arange(6) * 60
colors = plt.cm.hsv(torch.linspace(0, 5/6, 6).numpy())

fig_hidden, ax_hidden = plt.subplots(figsize=(9, 4), constrained_layout=True)
image = ax_hidden.imshow(hidden_by_rotation, aspect='auto', cmap='coolwarm')
ax_hidden.axvline(5.5, color='white', linewidth=2)
ax_hidden.set(xticks=range(12), yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='regular component (copies 1 | 2)', ylabel='input rotation', title='Hidden 2 Reg(C6) after PointActiv')
fig_hidden.colorbar(image, ax=ax_hidden, shrink=0.8)
plt.show()

## Step 5: visualize the tensor-product output

The second tensor product couples the activated regular features with themselves and projects the result into one $xy$ vector irrep plus four trivial components. We reconstruct one physical 3D vector from the first three components and plot the remaining three scalars separately.

In [ ]:
fig_output = plt.figure(figsize=(12, 5), constrained_layout=True)
ax_vec = fig_output.add_subplot(1, 2, 1, projection='3d')
for k, (vector, color) in enumerate(zip(output_vectors, colors)):
    ax_vec.quiver(0, 0, 0, *vector.tolist(), color=color, linewidth=2, label=f'{60*k}°')
ax_vec.set(xlabel='x', ylabel='y', zlabel='z', title='TensorProduct output vector')
output_limit = max(1e-3, 1.15 * output_vectors.abs().max().item())
ax_vec.set_xlim(-output_limit, output_limit); ax_vec.set_ylim(-output_limit, output_limit); ax_vec.set_zlim(-output_limit, output_limit); ax_vec.set_box_aspect((1, 1, 1))
ax_vec.legend(ncols=2, fontsize=8)

ax_scalar = fig_output.add_subplot(1, 2, 2)
for channel in range(3):
    ax_scalar.plot(angles_deg, output_scalars[:, channel], marker='o', label=f'output scalar {channel+1}')
ax_scalar.set(xticks=angles_deg.tolist(), xlabel='C6 rotation', ylabel='value', title='TensorProduct output scalars')
ax_scalar.grid(alpha=0.3); ax_scalar.legend()
plt.show()